# Step 1 — Recreate and Freeze Graph-Derived Priors

This notebook completes **Step 1 only**.

Goal: rebuild graph signals from the OCEL logs and save them as a frozen artifact for later steps.

## What we do in this step (simple view)

1. Load all 5 municipality logs.
2. Build transition graphs (what activity follows what).
3. Find common transitions (backbone) and unusual ones (rare/branch).
4. Compute activity stage ranks (early → late position).
5. Save everything to one file: `./output/graph_priors.json`.

In [1]:
import json
from pathlib import Path
from collections import defaultdict, Counter

DATASET_DIR = Path('./dataset')
OUTPUT_DIR = Path('./output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MUNICIPALITIES = [1, 2, 3, 4, 5]

print('Dataset dir:', DATASET_DIR.resolve())
print('Output dir :', OUTPUT_DIR.resolve())

Dataset dir: C:\Users\Lenovo\Documents\React Projects\bureaucratic-workflow-analyzer\dataset
Output dir : C:\Users\Lenovo\Documents\React Projects\bureaucratic-workflow-analyzer\output


In [2]:
def load_ocel(path: Path) -> dict:
    with open(path, 'r') as f:
        return json.load(f)

logs = {}
for m in MUNICIPALITIES:
    path = DATASET_DIR / f'BPIC15_Municipality{m}.jsonocel'
    logs[m] = load_ocel(path)
    events = logs[m]['ocel:events']
    objects = logs[m]['ocel:objects']
    print(f'Municipality {m}: {len(events):,} events | {len(objects):,} objects')

Municipality 1: 52,217 events | 1,269 objects
Municipality 2: 44,354 events | 859 objects
Municipality 3: 59,681 events | 1,465 objects
Municipality 4: 47,293 events | 1,084 objects
Municipality 5: 59,083 events | 1,202 objects


## Build the transition graph and stage ranks

- **Transition graph (DFG):** count each direct activity jump `A → B` inside each `Application` case.
- **Stage rank:** average normalized position of an activity in a case (`0 = early`, `1 = late`).

In [3]:
# Builds directly-follows graphs for each municipality
# It outputs a Counter of edges
# Each edge is a tuple (src_activity, tgt_activity) 
# The count is the number of times this transition occurs in the log
def build_dfg(log: dict, object_type: str = 'Application') -> Counter:
    events = log['ocel:events']
    objects = log['ocel:objects']

    target_oids = {oid for oid, obj in objects.items() if obj['ocel:type'] == object_type}

    obj_events = defaultdict(list)
    for event in events.values():
        ts = event['ocel:timestamp']
        act = event['ocel:activity']
        for oid in event.get('ocel:omap', []):
            if oid in target_oids:
                obj_events[oid].append((ts, act))

    dfg = Counter()
    for evs in obj_events.values():
        evs.sort(key=lambda x: x[0])
        for i in range(len(evs) - 1):
            src = evs[i][1]
            tgt = evs[i + 1][1]
            if src != tgt:
                dfg[(src, tgt)] += 1
    return dfg


# Computes activity ranks for each municipality
# Ranks are computed based on the average position of each activity in the case event sequences
# For each case (object of type 'Application'), we sort its events by timestamp
# Ranks are normalized to be between 0 and 1, where 0 means the activity always occurs first and 1 means it always occurs last in the cases
def compute_activity_ranks(log: dict, object_type: str = 'Application') -> dict:
    events = log['ocel:events']
    objects = log['ocel:objects']

    target_oids = {oid for oid, obj in objects.items() if obj['ocel:type'] == object_type}

    obj_events = defaultdict(list)
    for event in events.values():
        ts = event['ocel:timestamp']
        act = event['ocel:activity']
        for oid in event.get('ocel:omap', []):
            if oid in target_oids:
                obj_events[oid].append((ts, act))

    rank_sum = defaultdict(float)
    rank_count = defaultdict(int)

    for evs in obj_events.values():
        evs.sort(key=lambda x: x[0])
        n = len(evs)
        if n < 2:
            continue
        for idx, (_, act) in enumerate(evs):
            rank_sum[act] += idx / (n - 1)
            rank_count[act] += 1

    return {act: rank_sum[act] / rank_count[act] for act in rank_sum}


dfgs = {m: build_dfg(logs[m]) for m in MUNICIPALITIES}
act_ranks = {m: compute_activity_ranks(logs[m]) for m in MUNICIPALITIES}

for m in MUNICIPALITIES:
    top_edge = dfgs[m].most_common(1)[0] if dfgs[m] else None
    print(f'M{m}: {len(dfgs[m]):,} transitions | top edge: {top_edge}')

M1: 3,944 transitions | top edge: (('forward to the competent authority', 'regular procedure without MER'), 1063)
M2: 4,178 transitions | top edge: (('forward to the competent authority', 'regular procedure without MER'), 633)
M3: 4,280 transitions | top edge: (('procedure change', 'request complete'), 1127)
M4: 3,153 transitions | top edge: (('grounds for refusal', 'ask stakeholders views'), 760)
M5: 4,197 transitions | top edge: (('grounds for refusal', 'ask stakeholders views'), 866)


In [5]:
# Use top-K transitions per municipality to compare patterns
TOP_K = 100
BACKBONE_MIN_SUPPORT = 4  # edge appears in at least 4 of 5 municipalities
RARE_THRESHOLD = 0.2  # edge is in top-K of a municipality but not in others

top_sets = {m: set(edge for edge, _ in dfgs[m].most_common(TOP_K)) for m in MUNICIPALITIES}

support_count = Counter()
for m in MUNICIPALITIES:
    support_count.update(top_sets[m])

backbone_edges = [
    edge for edge, support in support_count.items()
    if support >= BACKBONE_MIN_SUPPORT
]

rare_edges_by_municipality = {}
for m in MUNICIPALITIES:
    others = set().union(*[top_sets[x] for x in MUNICIPALITIES if x != m])
    rare_edges_by_municipality[m] = sorted(list(top_sets[m] - others))

print(f'Backbone edges (support >= {BACKBONE_MIN_SUPPORT}): {len(backbone_edges)}')
for m in MUNICIPALITIES:
    print(f'M{m} rare edges: {len(rare_edges_by_municipality[m])}')

Backbone edges (support >= 4): 69
M1 rare edges: 19
M2 rare edges: 9
M3 rare edges: 10
M4 rare edges: 24
M5 rare edges: 9


## Freeze priors to disk

We now save a compact JSON artifact so later notebooks use the **same stable priors**.

Saved file: `./output/graph_priors.json`

In [6]:
def edge_to_obj(edge, weight=None):
    obj = {'src': edge[0], 'tgt': edge[1]}
    if weight is not None:
        obj['weight'] = int(weight)
    return obj

graph_priors = {
    'config': {
        'top_k': TOP_K,
        'backbone_min_support': BACKBONE_MIN_SUPPORT,
        'municipalities': MUNICIPALITIES
    },
    'backbone_edges': [edge_to_obj(edge) for edge in sorted(backbone_edges)],
    'rare_edges_by_municipality': {
        str(m): [edge_to_obj(edge) for edge in rare_edges_by_municipality[m]]
        for m in MUNICIPALITIES
    },
    'top_transitions_by_municipality': {
        str(m): [edge_to_obj(edge, weight) for edge, weight in dfgs[m].most_common(30)]
        for m in MUNICIPALITIES
    },
    'activity_rank_by_municipality': {
        str(m): {act: float(rank) for act, rank in sorted(act_ranks[m].items())}
        for m in MUNICIPALITIES
    }
}

out_path = OUTPUT_DIR / 'graph_priors.json'
with open(out_path, 'w') as f:
    json.dump(graph_priors, f, indent=2)

print('Saved:', out_path.resolve())

Saved: C:\Users\Lenovo\Documents\React Projects\bureaucratic-workflow-analyzer\output\graph_priors.json


In [7]:
# Quick sanity preview
print('Backbone preview (first 10):')
for row in graph_priors['backbone_edges'][:10]:
    print(' ', row['src'], '->', row['tgt'])

print('\nRare-edge preview per municipality (up to 5 each):')
for m in MUNICIPALITIES:
    sample = graph_priors['rare_edges_by_municipality'][str(m)][:5]
    total = len(graph_priors['rare_edges_by_municipality'][str(m)])
    print(f'M{m}: {total} total')
    for row in sample:
        print('   ', row['src'], '->', row['tgt'])

Backbone preview (first 10):
  OLO messaging active -> send confirmation receipt
  WAW permit aspect -> treat subcases content
  applicant is stakeholder -> terminate on request
  article 34  WABO applies -> assessment of content completed
  article 34  WABO applies -> grounds for refusal
  ask stakeholders views -> suspension ground applicable
  assessment of content completed -> grounds for refusal
  assessment of content completed -> phase advice known
  by law -> creating environmental permit decision
  by law -> decision date prior to decision

Rare-edge preview per municipality (up to 5 each):
M1: 19 total
    create subcases completeness -> create publication document
    creating environmental permit decision -> set decision status
    decision date prior to decision -> generating decision environmental permit
    enter senddate decision environmental permit -> enter date publication decision environmental permit
    enter senddate decision environmental permit -> start decisio

## Step 1 complete

You now have frozen graph priors that Step 2 can reuse directly.

Next notebook will build the case-step feature table from these logs and priors.